# Exploratory Data Analysis — TCGA-LUAD Lung Adenocarcinoma

Multi-omics dataset containing three normalised modalities:
- **RNASeq** — gene expression (447 samples × 14 434 genes)
- **DNAm** — DNA methylation β-values (420 samples × 384 629 CpG sites)
- **CNV** — copy-number variation (424 samples × 14 434 genes)
- **metadata** — sample annotations (846 entries)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

DATA_DIR = 'Group2-TCGA-LUAD-lung-adeno/'
print('Libraries loaded.')

## 1. Load data

In [ ]:
meta   = pd.read_csv(DATA_DIR + 'metadata.csv')
rnaseq = pd.read_csv(DATA_DIR + 'RNASeq.csv', index_col=0)
dnam   = pd.read_csv(DATA_DIR + 'DNAm.csv',   index_col=0)
cnv    = pd.read_csv(DATA_DIR + 'CNV.csv',    index_col=0)

print('metadata :', meta.shape)
print('RNASeq   :', rnaseq.shape)
print('DNAm     :', dnam.shape)
print('CNV      :', cnv.shape)

**Transform the name of the data**

1. Different connectors in sample ids:
    - For meta table, dash `_` is used to connect different part of the sample id/barcode (e.g. `TCGA-69-7765`)
    - For all three omics data, dot `.` is used as connector (e.g. `TCGA.69.7765`)
    - So step1 we need to replace all the dot connector `.` into dash `_`
2. Different length of sample ids:
    - For meta table, we have two types of sample ids: short barcode (e.g. `TCGA-69-7765`) and long barcode (e.g. `TCGA-50-5932-11A`)
    - In RNA seq data, we have two types of sample ids: short barcode which can match directly to meta table (e.g. `TCGA-69-7765`) and full length barcode (e.g. `TCGA-50-5932-11A-01R-1755-07`). So we clip length barcodes into the same lenghth in meta table.
    - In DNA methylation data, we only have full length barcode (e.g. `TCGA-69-7765-01A-11D-2168-05` and `TCGA-50-5932-11A-01D-1756-05`). So we clip full length barcodes into the same lenghth in meta table.
    - In CNV data, we have short barcode and long barcode which are the same as meta table, so we don't need further adjustment.

In [ ]:
# change sample id to match the format in meta table
rnaseq.index = rnaseq.index.str.replace(".", "-", regex=False)
rnaseq.index = rnaseq.index.str.split('-').str[:4].str.join('-')
print(f'Gene matrix: {rnaseq.shape[0]:,} samples x {rnaseq.shape[1]} genes')
rnaseq.iloc[:5, :5]

In [ ]:
dnam.index = dnam.index.str.replace(".", "-", regex=False)
print(f'DNAm matrix: {dnam.shape[0]:,} samples x {dnam.shape[1]} dnam')
dnam.iloc[:5, :5]

Sample id after clipping:

In [ ]:
dnam.index = dnam.index.str.split('-').str[:4].str.join('-')
print(f'DNAm matrix: {dnam.shape[0]:,} samples x {dnam.shape[1]} dnam')
dnam.iloc[:5, :5]

In [ ]:
cnv.index = cnv.index.str.replace(".", "-", regex=False)
print(f'CNV matrix: {cnv.shape[0]:,} samples x {cnv.shape[1]} cnv')
cnv.iloc[:5, :5]